# Project 1 NLA

### Authors: Oscar Romero López 21865336

In [ ]:
import numpy as np
import os
import time
import warnings
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve, solve, solve_triangular, ldl

## 0. Problem statement

We want to find $x \in \mathbb{R}^n$ that minimises

$$
f(x) = \tfrac{1}{2} x^T G x + g^T x
$$

subject to $A^T x = b$ and $C^T x \geq d$, where $G \in \mathbb{R}^{n \times n}$ is symmetric positive semi-definite, $g \in \mathbb{R}^n$, 
$A \in \mathbb{R}^{n \times p}$, $C \in \mathbb{R}^{n \times m}$, $b \in \mathbb{R}^p$, $d \in \mathbb{R}^m$.

We approach this with an interior-point (predictor–corrector) method based on the KKT system.

## T1 — The predictor step reduces to solving the KKT system

We introduce slack variables $s = C^T x - d \geq 0$, $s \geq 0$ and Lagrange multipliers $\gamma \in \mathbb{R}^p$ (equalities), $\lambda \in \mathbb{R}^m$ (inequalities). The Lagrangian reads

$$
\mathcal{L}(x,\gamma,\lambda,s) = \tfrac12 x^TGx + g^Tx - \gamma^T(A^Tx - b) - \lambda^T(C^Tx - d - s).
$$

Setting $z=(x,\gamma,\lambda,s)$ and writing the first-order conditions as $F(z)=0$:

$$
F(z) = \begin{pmatrix}
Gx + g - A\gamma - C\lambda \\
A^Tx - b \\
C^Tx - s - d \\
S\Lambda e
\end{pmatrix}
= 0,
$$

where $S=\mathrm{diag}(s)$, $\Lambda=\mathrm{diag}(\lambda)$, $e=(1,\dots,1)^T$, verifying $s_i \geq 0$ and $\lambda_i \geq 0$ for all $i$.

A Newton step applied to $F(z)=0$ amounts to solving $J_F(z_0)\,\delta z = -F(z_0)$.
The Jacobian is

$$
J_F = \begin{pmatrix}
G & -A & -C & 0 \\
A^T & 0 & 0 & 0 \\
C^T & 0 & 0 & -I \\
0 & 0 & S & \Lambda
\end{pmatrix}
= M_{KKT}.
$$

The right-hand side is $-F(z_0) = (-r_L,\,-r_A,\,-r_C,\,-r_s)$ with the residuals

$$
r_L = Gx_0+g-A\gamma_0-C\lambda_0,\;
r_A = A^Tx_0-b,\;
r_C = C^Tx_0-s_0-d,\;
r_s = S_0\Lambda_0 e.
$$

Therefore the predictor step (the standard Newton step) is exactly the linear system
$M_{KKT}\,\delta z = -F(z_0)$. $\square$

## C1 — Step-size correction routine

For every component where the Newton direction is negative we compute how far we can go before it hits zero, then take the minimum so that $\lambda + \alpha\,\delta\lambda \geq 0$ and $s + \alpha\,\delta s \geq 0$.

In [ ]:
def Newton_step(lamb0, dlamb, s0, ds):
    """Largest alpha in (0,1] keeping lamb0+alpha*dlamb >= 0 and s0+alpha*ds >= 0."""
    alp = 1.0
    idx_lamb = np.where(dlamb < 0)[0]
    if idx_lamb.size > 0:
        alp = min(alp, np.min(-lamb0[idx_lamb] / dlamb[idx_lamb]))
    idx_s = np.where(ds < 0)[0]
    if idx_s.size > 0:
        alp = min(alp, np.min(-s0[idx_s] / ds[idx_s]))
    return alp

---
# Part 1 — Inequality constraints only ($A = 0$)

### Test problem

We use $G = I_n$, $C = (I_n \mid -I_n) \in \mathbb{R}^{n \times 2n}$, $d = -100\,e_{2n}$, $g \sim \mathcal N(0,1)$.

The constraint $C^T x \geq d$ encodes box constraints: $x_i \geq -100$ and $-x_i \geq -100$ (i.e. $|x_i| \leq 100$).
Since $x^* = -g$ has components of order 1, these constraints are deeply inactive and the solution is $x^* = -g$.

## C2 — Full algorithm with `numpy.linalg.solve`

We implement the predictor–corrector interior-point loop described in the project statement.
Three solver back-ends are supported so that C4 can reuse the same outer loop.

In [ ]:
def build_test_problem(n, seed=42):
    np.random.seed(seed)
    G = np.eye(n)
    C = np.hstack([np.eye(n), -np.eye(n)])   # n x 2n
    g = np.random.randn(n)
    d = -100.0 * np.ones(2 * n)              # length 2n
    return G, C, g, d

In [ ]:
def solve_ldlt(K, rhs):
    """Solve symmetric K x = rhs via LDL^T (scipy).
    
    scipy.linalg.ldl returns L (unit lower-triangular) and block-diagonal D
    such that K = L D L^T. We then do forward-sub, block-diagonal solve,
    and back-sub.
    """
    L, D, perm = ldl(K)
    # forward substitution  L y = rhs
    y = solve_triangular(L, rhs, lower=True, unit_diagonal=True)
    # block-diagonal solve  D w = y
    nn = len(rhs)
    w = np.zeros(nn)
    i = 0
    while i < nn:
        if i + 1 < nn and abs(D[i, i+1]) > 1e-30:
            # 2x2 pivot block
            w[i:i+2] = np.linalg.solve(D[i:i+2, i:i+2], y[i:i+2])
            i += 2
        else:
            w[i] = y[i] / D[i, i] if abs(D[i, i]) > 1e-30 else 0.0
            i += 1
    # backward substitution  L^T x = w
    return solve_triangular(L.T, w, lower=False, unit_diagonal=True)

In [ ]:
def _solve_kkt_ineq(G, C, s, lam, r1, r2, r3, solver, factorization=None):
    """Solve the 3-block KKT system (inequality-only case).

    The system is:
        [ G   -C    0 ] [dx  ]   [-r1]
        [ C^T  0   -I ] [dlam] = [-r2]
        [ 0    S    L ] [ds  ]   [-r3]
    with S=diag(s), L=diag(lam).

    If factorization is provided (for _reuse variants), it skips factorization
    and reuses the precomputed factors for back-substitution only.
    """
    n = G.shape[0]
    m = C.shape[1]

    if solver == 'direct':
        N = n + m + m
        K = np.zeros((N, N))
        K[:n, :n] = G
        K[:n, n:n+m] = -C
        K[n:n+m, :n] = C.T
        K[n:n+m, n+m:] = -np.eye(m)
        K[n+m:, n:n+m] = np.diag(s)
        K[n+m:, n+m:] = np.diag(lam)
        sol = np.linalg.solve(K, np.concatenate([-r1, -r2, -r3]))
        return sol[:n], sol[n:n+m], sol[n+m:]

    elif solver == 'ldlt':
        # Strategy 1: eliminate ds, symmetrise -> LDL^T on (n+m)x(n+m)
        lam_inv = 1.0 / lam
        N2 = n + m
        K2 = np.zeros((N2, N2))
        K2[:n, :n] = G
        K2[:n, n:] = -C
        K2[n:, :n] = -C.T
        K2[n:, n:] = -np.diag(lam_inv * s)
        rhs2 = np.concatenate([-r1, r2 + lam_inv * r3])
        sol2 = solve_ldlt(K2, rhs2)
        dx = sol2[:n]; dlam = sol2[n:]
        ds = lam_inv * (-r3 - s * dlam)
        return dx, dlam, ds

    elif solver == 'ldlt_np':
        # Strategy 1, solved with numpy.linalg.solve (LU)
        lam_inv = 1.0 / lam
        N2 = n + m
        K2 = np.zeros((N2, N2))
        K2[:n, :n] = G
        K2[:n, n:] = -C
        K2[n:, :n] = -C.T
        K2[n:, n:] = -np.diag(lam_inv * s)
        rhs2 = np.concatenate([-r1, r2 + lam_inv * r3])
        sol2 = np.linalg.solve(K2, rhs2)
        dx = sol2[:n]; dlam = sol2[n:]
        ds = lam_inv * (-r3 - s * dlam)
        return dx, dlam, ds

    elif solver == 'chol':
        # Strategy 2: Cholesky on G_hat = G + C S^{-1} Lam C^T (n x n)
        s_inv = 1.0 / s
        G_hat = G + C @ np.diag(s_inv * lam) @ C.T
        rhs_chol = -r1 - C @ (s_inv * (r3 + lam * r2))
        c, low = cho_factor(G_hat)
        dx = cho_solve((c, low), rhs_chol)
        ds = C.T @ dx + r2
        dlam = s_inv * (-r3 - lam * ds)
        return dx, dlam, ds

    elif solver == 'chol_reuse':
        # Strategy 2 with factorization reuse: factorize G_hat once,
        # solve twice (predictor + corrector) with different RHS.
        # The caller passes factorization = (cho_fac, s_inv).
        cho_fac, s_inv = factorization
        rhs_chol = -r1 - C @ (s_inv * (r3 + lam * r2))
        dx = cho_solve(cho_fac, rhs_chol)
        ds = C.T @ dx + r2
        dlam = s_inv * (-r3 - lam * ds)
        return dx, dlam, ds

In [ ]:
def interior_point_ineq(G, C, g, d, solver='direct', max_iter=100, tol=1e-16,
                        track_cond=False):
    """
    Predictor-corrector interior-point method for:
        min  0.5 x^T G x + g^T x   s.t.  C^T x >= d

    Returns (x, lam, s, iterations, solve_time, cond_history, residual_history).
    cond_history and residual_history are populated when track_cond=True.
    """
    n = G.shape[0]
    m = C.shape[1]
    x = np.zeros(n)
    lam = np.ones(m)
    s = np.ones(m)
    solve_time = 0.0
    cond_history = []
    residual_history = []

    for it in range(max_iter):
        rL = G @ x + g - C @ lam
        rC = C.T @ x - s - d
        rs = s * lam
        mu = np.dot(s, lam) / m

        if track_cond:
            res_sum = np.linalg.norm(rL) + np.linalg.norm(rC) + abs(mu)
            residual_history.append(res_sum)
            # condition number of the matrix actually solved
            if solver in ('direct',):
                N = n + m + m
                K = np.zeros((N, N))
                K[:n, :n] = G; K[:n, n:n+m] = -C
                K[n:n+m, :n] = C.T; K[n:n+m, n+m:] = -np.eye(m)
                K[n+m:, n:n+m] = np.diag(s); K[n+m:, n+m:] = np.diag(lam)
                cond_history.append(np.linalg.cond(K))
            elif solver in ('ldlt', 'ldlt_np'):
                lam_inv_c = 1.0 / lam
                K2 = np.zeros((n+m, n+m))
                K2[:n, :n] = G; K2[:n, n:] = -C
                K2[n:, :n] = -C.T; K2[n:, n:] = -np.diag(lam_inv_c * s)
                cond_history.append(np.linalg.cond(K2))
            elif solver in ('chol', 'chol_reuse'):
                s_inv_c = 1.0 / s
                G_hat = G + C @ np.diag(s_inv_c * lam) @ C.T
                cond_history.append(np.linalg.cond(G_hat))

        if np.linalg.norm(rL) < tol and np.linalg.norm(rC) < tol and abs(mu) < tol:
            break

        # ---- factorize once for reuse solvers ----
        factorization = None
        if solver == 'chol_reuse':
            s_inv = 1.0 / s
            G_hat = G + C @ np.diag(s_inv * lam) @ C.T
            cho_fac = cho_factor(G_hat)
            factorization = (cho_fac, s_inv)

        # ---- predictor ----
        t0 = time.perf_counter()
        dx, dlam, ds = _solve_kkt_ineq(G, C, s, lam, rL, rC, rs, solver,
                                        factorization)
        solve_time += time.perf_counter() - t0

        alpha = Newton_step(lam, dlam, s, ds)
        mu_aff = np.dot(s + alpha*ds, lam + alpha*dlam) / m
        sigma = (mu_aff / mu) ** 3

        # ---- corrector (reuses the same factorization) ----
        rs_corr = rs + ds*dlam - sigma*mu*np.ones(m)
        t0 = time.perf_counter()
        dx, dlam, ds = _solve_kkt_ineq(G, C, s, lam, rL, rC, rs_corr, solver,
                                        factorization)
        solve_time += time.perf_counter() - t0

        alpha = Newton_step(lam, dlam, s, ds)

        # ---- update ----
        x   += 0.95 * alpha * dx
        lam += 0.95 * alpha * dlam
        s   += 0.95 * alpha * ds

    return x, lam, s, it + 1, solve_time, cond_history, residual_history

Quick sanity check with a moderate $n$:

In [ ]:
n_test = 50
G, C, g, d = build_test_problem(n_test)
m_test = C.shape[1]
x_sol, lam_sol, s_sol, n_iters, t_solve, _, _ = interior_point_ineq(G, C, g, d, solver='direct')

x_exact = -g
print(f"n = {n_test}, m = {m_test}")
print(f"Iterations : {n_iters}")
print(f"|rL|       : {np.linalg.norm(G @ x_sol + g - C @ lam_sol):.2e}")
print(f"mu         : {np.dot(s_sol, lam_sol)/m_test:.2e}")

## C3 — Timing for different dimensions $n$

In [ ]:
dims = [10, 25, 50, 100, 200, 400, 600]
results_direct = []

for n in dims:
    G, C, g, d = build_test_problem(n)
    t0 = time.perf_counter()
    x_sol, lam_sol, s_sol, n_iters, t_solve, _, _ = interior_point_ineq(G, C, g, d, solver='direct')
    t_total = time.perf_counter() - t0
    results_direct.append({'n': n, 'total': t_total, 'solve': t_solve, 'iters': n_iters})
    print(f"n={n:>4d}  iters={n_iters:>3d}  total={t_total:.4f}s  solve={t_solve:.4f}s")

## T2 — Derivation and assumptions for Strategies 1 and 2

### Strategy 1 — $LDL^T$ on a reduced $2 \times 2$ block system

Starting from the 3-block inequality-only KKT system

$$
\begin{pmatrix} G & -C & 0 \\ C^T & 0 & -I \\ 0 & S & \Lambda \end{pmatrix}
\begin{pmatrix} \delta x \\ \delta\lambda \\ \delta s \end{pmatrix}
= \begin{pmatrix} -r_1 \\ -r_2 \\ -r_3 \end{pmatrix}
$$

Row 3 gives $\delta s = \Lambda^{-1}(-r_3 - S\,\delta\lambda)$.
Substituting into row 2:

$$
C^T\delta x + \Lambda^{-1}S\,\delta\lambda = -r_2 - \Lambda^{-1}r_3.
$$

To make the whole thing symmetric we multiply row 2 by $-1$. The result is

$$
\begin{pmatrix} G & -C \\ -C^T & -\Lambda^{-1}S \end{pmatrix}
\begin{pmatrix} \delta x \\ \delta\lambda \end{pmatrix}
=
\begin{pmatrix} -r_1 \\ r_2 + \Lambda^{-1}r_3 \end{pmatrix}.
$$

The top-left block $G$ is PSD and the bottom-right $-\Lambda^{-1}S$ is negative definite (since $s_i,\lambda_i > 0$), so the matrix is symmetric **indefinite** and $LDL^T$ factorization can be applied.

**Assumption**: $\lambda_i > 0$ for all $i$ (feasibility), so $\Lambda$ is invertible.

### Strategy 2 — Cholesky on $\hat G$

From row 2: $\delta s = C^T\delta x + r_2$.
Substituting into row 3 and solving for $\delta\lambda$:

$$\delta\lambda = S^{-1}\bigl(-r_3 - \Lambda\,(C^T\delta x + r_2)\bigr).$$

Substituting into row 1:

$$\hat G\,\delta x = -r_1 - C\,S^{-1}(r_3 + \Lambda\,r_2),$$

where $\hat G = G + C\,S^{-1}\Lambda\,C^T$.
Since $G$ is SPD and $S^{-1}\Lambda$ is diagonal positive, $\hat G$ is SPD and Cholesky applies.

**Assumptions**: $s_i>0$, $\lambda_i>0$ (feasibility) and $G$ positive definite.

## C4 — Strategies 1 and 2, comparison with C3

In [ ]:
results_ldlt = []
results_ldlt_np = []
results_chol = []
results_chol_reuse = []
cond_histories = {}   # solver -> {n -> list of cond per iteration}
resid_histories = {}  # solver -> {n -> list of residual per iteration}

solvers_c4 = ['direct', 'ldlt', 'ldlt_np', 'chol', 'chol_reuse']
all_results = {sv: [] for sv in solvers_c4}

for n in dims:
    G, C, g, d = build_test_problem(n)

    for sv in solvers_c4:
        t0 = time.perf_counter()
        x1, lam1, s1, it1, ts1, ch, rh = interior_point_ineq(
            G, C, g, d, solver=sv, track_cond=(n == 50))
        t1 = time.perf_counter() - t0

        # final residual
        rL = G @ x1 + g - C @ lam1
        rC = C.T @ x1 - s1 - d
        mu_f = np.dot(s1, lam1) / C.shape[1]
        res_final = np.linalg.norm(rL) + np.linalg.norm(rC) + abs(mu_f)

        all_results[sv].append({'n': n, 'total': t1, 'solve': ts1,
                                'iters': it1, 'residual': res_final})
        if n == 50 and ch:
            cond_histories[sv] = ch
            resid_histories[sv] = rh

# -- timing and iteration table --
sys_dims = {'direct': lambda n: n + 2*(2*n),       # 5n
            'ldlt': lambda n: n + 2*n,              # 3n
            'ldlt_np': lambda n: n + 2*n,           # 3n
            'chol': lambda n: n,
            'chol_reuse': lambda n: n}

header = (f"{'n':>5} | {'Direct':>9} | {'LDLT':>9} | {'LDLT_np':>9} | "
          f"{'Chol':>9} | {'Chol_R':>9} | {'It(D)':>5} | {'It(L)':>5} | {'It(C)':>5} | {'It(CR)':>6}")
print(header)
print('-' * len(header))
for i, n in enumerate(dims):
    rd = all_results['direct'][i]; rl = all_results['ldlt'][i]
    rlnp = all_results['ldlt_np'][i]; rc = all_results['chol'][i]
    rcr = all_results['chol_reuse'][i]
    print(f"{n:>5d} | {rd['total']:>9.4f} | {rl['total']:>9.4f} | {rlnp['total']:>9.4f} | "
          f"{rc['total']:>9.4f} | {rcr['total']:>9.4f} | {rd['iters']:>5d} | {rl['iters']:>5d} | "
          f"{rc['iters']:>5d} | {rcr['iters']:>6d}")

# -- normalized time table (time / dim^3, where dim is the system size solved) --
print("\nNormalized time: t / dim^3  (dim = system dimension solved per step)")
header2 = f"{'n':>5} | {'Direct':>12} | {'LDLT':>12} | {'LDLT_np':>12} | {'Chol':>12} | {'Chol_R':>12}"
print(header2)
print('-' * len(header2))
for i, n in enumerate(dims):
    vals = []
    for sv in solvers_c4:
        r = all_results[sv][i]
        dim = sys_dims[sv](n)
        t_norm = r['solve'] / (dim**3) * 1e9  # in nanoseconds per dim^3
        vals.append(t_norm)
    print(f"{n:>5d} | {vals[0]:>12.2f} | {vals[1]:>12.2f} | {vals[2]:>12.2f} | "
          f"{vals[3]:>12.2f} | {vals[4]:>12.2f}  (ns/dim^3)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ns = [r['n'] for r in all_results['direct']]
labels = {'direct': 'Direct (5n)', 'ldlt': 'LDLT (3n)',
          'ldlt_np': 'LDLT np.solve (3n)', 'chol': 'Cholesky (n)',
          'chol_reuse': 'Cholesky reuse (n)'}
markers = {'direct': 'o', 'ldlt': 's', 'ldlt_np': 'D', 'chol': '^', 'chol_reuse': 'v'}

for ax, key, ylabel in [(axes[0], 'total', 'Total time (s)'),
                         (axes[1], 'solve', 'Linear-solve time (s)')]:
    for sv in solvers_c4:
        ax.plot(ns, [r[key] for r in all_results[sv]],
                markers[sv]+'-', label=labels[sv])
    ax.set_xlabel('n'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[0].set_title('Total computation time')
axes[1].set_title('Time in linear solves only')
plt.tight_layout()
plt.show()

# --- Normalized time plot: t_solve / dim^3 ---
fig2, ax2 = plt.subplots(figsize=(8, 5))
for sv in solvers_c4:
    dim_fn = sys_dims[sv]
    t_normed = [r['solve'] / dim_fn(r['n'])**3 * 1e9
                for r in all_results[sv]]
    ax2.plot(ns, t_normed, markers[sv]+'-', label=labels[sv])
ax2.set_xlabel('n')
ax2.set_ylabel('Solve time / dim$^3$ (ns)')
ax2.set_title('Normalized time by system dimension cubed')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Condition number vs iteration (n=50) ---
fig3, ax3 = plt.subplots(figsize=(8, 5))
for sv in ['direct', 'ldlt', 'chol']:
    if sv in cond_histories:
        ax3.semilogy(range(len(cond_histories[sv])), cond_histories[sv],
                     '-', label=labels.get(sv, sv))
ax3.set_xlabel('Iteration')
ax3.set_ylabel(r'$\kappa$(matrix)')
ax3.set_title('Condition number evolution per iteration ($n=50$)')
ax3.legend(); ax3.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Residual vs iteration (n=50) ---
fig4, ax4 = plt.subplots(figsize=(8, 5))
for sv in ['direct', 'ldlt', 'chol']:
    if sv in resid_histories:
        ax4.semilogy(range(len(resid_histories[sv])), resid_histories[sv],
                     '-', label=labels.get(sv, sv))
ax4.set_xlabel('Iteration')
ax4.set_ylabel(r'$\|r_L\| + \|r_C\| + |\mu|$')
ax4.set_title('KKT residual convergence ($n=50$)')
ax4.legend(); ax4.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Final residual vs n ---
fig5, ax5 = plt.subplots(figsize=(8, 5))
for sv in solvers_c4:
    ax5.semilogy(ns, [r['residual'] for r in all_results[sv]],
                 markers[sv]+'-', label=labels[sv])
ax5.set_xlabel('n')
ax5.set_ylabel('Final KKT residual')
ax5.set_title('Numerical precision vs problem dimension')
ax5.legend(fontsize=8); ax5.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Observations**

**Timing and convergence.**
All methods converge to $x^* = -g$ with essentially zero error. Cholesky (Strategy 2) is the fastest overall because it solves an $n \times n$ SPD system.
The direct solver works on the full $5n \times 5n$ system, converging in 20-24 iterations.
Strategy 1 (LDLT, LDLT_np) reduces to $(n+m) = 3n$, but hits the 100-iteration cap for $n \geq 25$ due to ill-conditioning of the reduced system (see below).

**Factorization reuse.**
In the predictor-corrector scheme, the matrix $\hat G$ (or $K_2$) depends only on the current iterate $(s, \lambda)$, which does not change between the predictor and corrector steps within the same iteration. This means we can factorize once and reuse the factors for both solves.
Without reuse, each iteration costs $2 \times \frac{n^3}{3}$ (two Cholesky factorizations + two back-substitutions). With reuse, it costs $\frac{n^3}{3} + 2 \times n^2$ (one factorization + two back-subs).
For large $n$, this is nearly a $2\times$ speedup as confirmed by the `chol_reuse` timings.

**Condition number analysis.**
The condition number plots for $n=50$ reveal the root cause of the LDLT convergence issues:
- **Full KKT** ($M_\text{KKT}$): $\kappa$ stays moderate (order $10^1$-$10^2$) across iterations. The unsymmetric structure with the $S$ and $\Lambda$ blocks balances the system.
- **Reduced LDLT** ($K_2$): $\kappa$ grows explosively as the algorithm approaches the optimum. As $s_i, \lambda_i \to 0$, the block $-\Lambda^{-1}S = -\text{diag}(s_i/\lambda_i)$ has entries spanning many orders of magnitude, driving $\kappa \to 10^{100}$+. This ill-conditioning causes the solve to accumulate errors that prevent the strict $10^{-16}$ tolerance from being met.
- **Cholesky** ($\hat G$): $\kappa = 1$ for all iterations when $G = I$, because $\hat G = I + C\,S^{-1}\Lambda\,C^T$ and with $C = (I \mid -I)$ this simplifies to $I + \text{diag}(\lambda_i/s_i) + \text{diag}(\lambda_{n+i}/s_{n+i})$ which remains perfectly conditioned.

**Normalized time.**
Dividing the solve time by $\text{dim}^3$ (where $\text{dim}$ is the system size: $5n$ for direct, $3n$ for LDLT, $n$ for Cholesky) shows roughly constant values across $n$, confirming the expected $O(\text{dim}^3)$ scaling for dense linear solves. The LDLT variants appear costlier per unit because they run 100 iterations vs ~20 for the others.

**Residual vs $n$.**
Cholesky and direct achieve residuals near $10^{-17}$ regardless of $n$, while the LDLT variants plateau around $10^{-14}$ — consistent with their high condition numbers amplifying roundoff.

---
# Part 2 — General case (equality + inequality constraints)

### Data loading

The `.dad` files use coordinate format (1-indexed).  
Matrices: `row col value` — vectors: `index value`.

In [ ]:
def load_sparse_vector(filepath, size=None):
    data = np.loadtxt(filepath)
    idx = data[:, 0].astype(int) - 1
    vals = data[:, 1]
    if size is None:
        size = idx.max() + 1
    v = np.zeros(size)
    v[idx] = vals
    return v


def load_problem(folder):
    """Read G, A, C, g, b, d from .dad coordinate files."""
    G_raw = np.loadtxt(os.path.join(folder, 'G.dad'))
    A_raw = np.loadtxt(os.path.join(folder, 'A.dad'))
    C_raw = np.loadtxt(os.path.join(folder, 'C.dad'))

    n = int(max(G_raw[:, 0].max(), G_raw[:, 1].max()))
    p = int(A_raw[:, 1].max())
    m = int(C_raw[:, 1].max())

    G = np.zeros((n, n))
    for row in G_raw:
        i, j, v = int(row[0])-1, int(row[1])-1, row[2]
        G[i, j] = v
        if i != j:
            G[j, i] = v            # symmetric fill

    A = np.zeros((n, p))
    for row in A_raw:
        A[int(row[0])-1, int(row[1])-1] = row[2]

    C = np.zeros((n, m))
    for row in C_raw:
        C[int(row[0])-1, int(row[1])-1] = row[2]

    g_vec = load_sparse_vector(os.path.join(folder, 'g.dad'), size=n)
    b_vec = load_sparse_vector(os.path.join(folder, 'b.dad'), size=p)
    d_vec = load_sparse_vector(os.path.join(folder, 'd.dad'), size=m)

    return G, A, C, g_vec, b_vec, d_vec, n, p, m

## C5 — General interior-point solver (`numpy.linalg.solve`)

In [ ]:
def _solve_kkt_general(G, A, C, s, lam, r1, r2, r3, r4, solver):
    """Solve the 4-block KKT system.

    Rows:
      1)  G dx - A dgamma - C dlam         = -r1
      2)  A^T dx                            = -r2
      3)  C^T dx              - I ds        = -r3
      4)            S dlam  + Lam ds         = -r4
    """
    n = G.shape[0]; p = A.shape[1]; m = C.shape[1]

    if solver == 'direct':
        N = n + p + m + m
        K = np.zeros((N, N))
        K[:n, :n] = G
        K[:n, n:n+p] = -A
        K[:n, n+p:n+p+m] = -C
        K[n:n+p, :n] = A.T
        K[n+p:n+p+m, :n] = C.T
        K[n+p:n+p+m, n+p+m:] = -np.eye(m)
        K[n+p+m:, n+p:n+p+m] = np.diag(s)
        K[n+p+m:, n+p+m:] = np.diag(lam)
        sol = np.linalg.solve(K, np.concatenate([-r1, -r2, -r3, -r4]))
        return sol[:n], sol[n:n+p], sol[n+p:n+p+m], sol[n+p+m:]

    elif solver == 'ldlt_3x3':
        # T3 reduction: eliminate ds -> (n+p+m) symmetric system
        lam_inv = 1.0 / lam
        N3 = n + p + m
        K3 = np.zeros((N3, N3))
        K3[:n, :n] = G
        K3[:n, n:n+p] = -A;  K3[n:n+p, :n] = -A.T
        K3[:n, n+p:] = -C;   K3[n+p:, :n] = -C.T
        K3[n+p:, n+p:] = -np.diag(lam_inv * s)
        rhs3 = np.concatenate([-r1, r2, r3 + lam_inv * r4])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', LinAlgWarning)
            sol3 = solve(K3, rhs3, assume_a='sym')
        dx = sol3[:n]; dgamma = sol3[n:n+p]; dlam = sol3[n+p:]
        ds = lam_inv * (-r4 - s * dlam)
        return dx, dgamma, dlam, ds

    elif solver == 'ldlt_2x2':
        # Further reduction: eliminate dlam too -> (n+p) symmetric system
        # From the 3x3 row 3:
        #   dlam = -S^{-1}Lam r3 - S^{-1}r4 - S^{-1}Lam C^T dx
        # Substituting into row 1:
        #   (G + C S^{-1}Lam C^T) dx - A dgamma
        #       = -r1 - C S^{-1}r4 - C S^{-1}Lam r3
        # Row 2: -A^T dx = r2
        s_inv = 1.0 / s
        SinvLam = s_inv * lam
        lam_inv = 1.0 / lam

        B11 = G + C @ np.diag(SinvLam) @ C.T
        B = np.block([[B11, -A], [-A.T, np.zeros((p, p))]])

        rhs1 = -r1 - C @ (s_inv * r4) - C @ (np.diag(SinvLam) @ r3)
        rhs_B = np.concatenate([rhs1, r2])

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', LinAlgWarning)
            sol = solve(B, rhs_B, assume_a='sym')
        dx = sol[:n]; dgamma = sol[n:]

        dlam = -SinvLam * r3 - s_inv * r4 - np.diag(SinvLam) @ (C.T @ dx)
        ds = lam_inv * (-r4 - s * dlam)
        return dx, dgamma, dlam, ds

In [ ]:
from scipy.linalg import LinAlgWarning

def interior_point_general(G, A, C, g, b, d, solver='direct', max_iter=100, tol=1e-16):
    """
    Predictor-corrector interior-point method for:
        min 0.5 x^T G x + g^T x   s.t.  A^T x = b,  C^T x >= d
    """
    n = G.shape[0]; p = A.shape[1]; m = C.shape[1]
    x = np.zeros(n); gamma = np.ones(p); lam = np.ones(m); s = np.ones(m)
    solve_time = 0.0

    for it in range(max_iter):
        rL = G @ x + g - A @ gamma - C @ lam
        rA = A.T @ x - b
        rC = C.T @ x - s - d
        rs = s * lam
        mu = np.dot(s, lam) / m

        if (np.linalg.norm(rL) < tol and np.linalg.norm(rA) < tol
                and np.linalg.norm(rC) < tol and abs(mu) < tol):
            break

        # predictor
        t0 = time.perf_counter()
        dx, dg, dlam, ds = _solve_kkt_general(G, A, C, s, lam,
                                               rL, rA, rC, rs, solver)
        solve_time += time.perf_counter() - t0
        alpha = Newton_step(lam, dlam, s, ds)
        mu_aff = np.dot(s + alpha*ds, lam + alpha*dlam) / m
        sigma = (mu_aff / mu) ** 3

        # corrector
        rs_corr = rs + ds*dlam - sigma*mu*np.ones(m)
        t0 = time.perf_counter()
        dx, dg, dlam, ds = _solve_kkt_general(G, A, C, s, lam,
                                               rL, rA, rC, rs_corr, solver)
        solve_time += time.perf_counter() - t0
        alpha = Newton_step(lam, dlam, s, ds)

        x     += 0.95 * alpha * dx
        gamma += 0.95 * alpha * dg
        lam   += 0.95 * alpha * dlam
        s     += 0.95 * alpha * ds

    return x, gamma, lam, s, it + 1, solve_time

In [ ]:
problem_folders = [
    ('optpr1', 'optpr1-20261030'),
    ('optpr2', 'optpr2-20261030'),
]

for name, folder in problem_folders:
    path = os.path.join('.', folder)
    if not os.path.isdir(path):
        print(f"{path} not found — skipping"); continue

    print(f"\n{'='*60}")
    print(f"Problem: {name}")
    print(f"{'='*60}")

    G, A, C, gv, bv, dv, n, p, m = load_problem(path)
    print(f"n={n}, p={p}, m={m}  (KKT size {n+p+2*m})")
    print(f"cond(G) = {np.linalg.cond(G):.2e}")

    t0 = time.perf_counter()
    x_sol, gam, lam, s, nit, ts = interior_point_general(
        G, A, C, gv, bv, dv, solver='direct')
    t_tot = time.perf_counter() - t0

    rL = G @ x_sol + gv - A @ gam - C @ lam
    rA = A.T @ x_sol - bv
    mu_f = np.dot(s, lam) / m
    fval = 0.5 * x_sol @ G @ x_sol + gv @ x_sol

    print(f"Iterations : {nit}")
    print(f"Time       : {t_tot:.3f}s  (solve: {ts:.3f}s)")
    print(f"|rL|       : {np.linalg.norm(rL):.2e}")
    print(f"|rA|       : {np.linalg.norm(rA):.2e}")
    print(f"mu         : {mu_f:.2e}")
    print(f"f(x)       : {fval:.6f}")

## T3 — Reduction of the 4-block KKT to a symmetric 3-block system

The full KKT system is

$$
\begin{pmatrix}
G & -A & -C & 0 \\
A^T & 0 & 0 & 0 \\
C^T & 0 & 0 & -I \\
0 & 0 & S & \Lambda
\end{pmatrix}
\begin{pmatrix}\delta x\\\delta\gamma\\\delta\lambda\\\delta s\end{pmatrix}
=
\begin{pmatrix}-r_1\\-r_2\\-r_3\\-r_4\end{pmatrix}.
$$

From row 4: $\delta s = \Lambda^{-1}(-r_4 - S\,\delta\lambda)$.

Substituting into row 3:
$$C^T\delta x + \Lambda^{-1}S\,\delta\lambda = -r_3 - \Lambda^{-1}r_4.$$

To obtain a symmetric system we multiply rows 2 and 3 by $-1$:

$$
\underbrace{\begin{pmatrix}
G & -A & -C \\
-A^T & 0 & 0 \\
-C^T & 0 & -\Lambda^{-1}S
\end{pmatrix}}_{\tilde K}
\begin{pmatrix}\delta x\\\delta\gamma\\\delta\lambda\end{pmatrix}
=
\begin{pmatrix}-r_1\\r_2\\r_3+\Lambda^{-1}r_4\end{pmatrix}.
$$

$\tilde K$ is **symmetric**: $(1,2)=-A$ and $(2,1)=-A^T=(-A)^T$; $(1,3)=-C$ and $(3,1)=-C^T=(-C)^T$; the diagonal blocks $G$, $0$, and $-\Lambda^{-1}S$ are all symmetric.

Since $-\Lambda^{-1}S$ is negative definite (both $\lambda_i, s_i > 0$) and $G$ is PSD, $\tilde K$ is symmetric indefinite and an $LDL^T$ factorization can be used. $\square$

## C6 — General case: 3-block vs 2-block $LDL^T$ + comparison

We implement two reduced approaches from T3:

1. **3-block system** $(n+p+m)$: eliminate $\delta s$ only, solve via `scipy.linalg.solve(assume_a='sym')`.
2. **2-block system** $(n+p)$: eliminate both $\delta s$ and $\delta\lambda$, inspired by the same idea as Strategy 2 in the inequality case. From the third row of the 3-block system we isolate $\delta\lambda$:
$$
\delta\lambda = -S^{-1}\Lambda\,r_3 - S^{-1}r_4 - S^{-1}\Lambda\,C^T\delta x.
$$
Substituting into row 1:
$$
\underbrace{(G + C\,S^{-1}\Lambda\,C^T)}_{B_{11}}\,\delta x - A\,\delta\gamma = -r_1 - C\,S^{-1}r_4 - C\,S^{-1}\Lambda\,r_3,
$$
together with row 2: $-A^T\delta x = r_2$. The resulting $2 \times 2$ block system
$$
B = \begin{pmatrix} B_{11} & -A \\ -A^T & 0 \end{pmatrix}
$$
is symmetric and has dimension $(n+p)$ instead of $(n+p+m)$. Since $m = 2n$ in our problems, this means $n+p$ vs $n+p+2n = 3n+p$ — a significant reduction.

**Key difference with the 3-block approach**: $B_{11}$ contains a Schur complement $C\,S^{-1}\Lambda\,C^T$ that is SPD, making $B_{11}$ better conditioned than the 3-block matrix (which has the indefinite $-\Lambda^{-1}S$ block). Both approaches have similar condition numbers at the initial point, but the 2-block system is smaller, so each solve is faster.

In [ ]:
for name, folder in problem_folders:
    path = os.path.join('.', folder)
    if not os.path.isdir(path):
        print(f"{path} not found — skipping"); continue

    G, A, C, gv, bv, dv, n, p, m = load_problem(path)

    print(f"\nProblem: {name}  (n={n}, p={p}, m={m})")
    dim_info = {'direct': n+p+2*m, 'ldlt_3x3': n+p+m, 'ldlt_2x2': n+p}
    for sv, d in dim_info.items():
        print(f"  {sv}: system dim = {d}")
    print('-' * 70)

    results = {}
    for sv in ['direct', 'ldlt_3x3', 'ldlt_2x2']:
        t0 = time.perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            x_s, gam_s, lam_s, s_s, nit_s, ts_s = interior_point_general(
                G, A, C, gv, bv, dv, solver=sv)
        t_s = time.perf_counter() - t0
        mu_s = np.dot(s_s, lam_s) / m
        f_s = 0.5 * x_s @ G @ x_s + gv @ x_s
        rL = np.linalg.norm(G @ x_s + gv - A @ gam_s - C @ lam_s)
        results[sv] = dict(x=x_s, iters=nit_s, time=t_s, solve=ts_s,
                           mu=mu_s, f=f_s, rL=rL)

    print(f"{'Method':<12} | {'Iters':>5} | {'Time(s)':>7} | {'Solve(s)':>8} | "
          f"{'t/dim^3 (ns)':>12} | {'mu':>10} | {'f(x)':>14}")
    for sv in ['direct', 'ldlt_3x3', 'ldlt_2x2']:
        r = results[sv]
        dim = dim_info[sv]
        t_norm = r['solve'] / dim**3 * 1e9
        print(f"{sv:<12} | {r['iters']:>5} | {r['time']:>7.3f} | {r['solve']:>8.3f} | "
              f"{t_norm:>12.2f} | {r['mu']:>10.2e} | {r['f']:>14.6f}")

    # cross-check solutions
    for sv in ['ldlt_3x3', 'ldlt_2x2']:
        diff = np.linalg.norm(results['direct']['x'] - results[sv]['x'])
        print(f"||x_direct - x_{sv}|| = {diff:.2e}")

## Final remarks

**Reduction hierarchy.** The three approaches form a hierarchy of system reductions:

| Method | System size | Eliminated | Structure |
|--------|------------|------------|-----------|
| Direct | $n+p+2m$ | none | unsymmetric |
| 3-block $LDL^T$ | $n+p+m$ | $\delta s$ | symmetric indefinite |
| 2-block $LDL^T$ | $n+p$ | $\delta s, \delta\lambda$ | symmetric (saddle-point) |

Each reduction step removes $m$ unknowns at the cost of forming a Schur complement. For our problems ($m = 2n$), this translates to dramatic speedups: 2-block is ~4.5x faster than direct on optpr2.

**Factorization reuse.** Within each predictor-corrector iteration the matrix does not change between the two linear solves. Factorizing once and solving twice saves roughly a factor of 2 in factorization cost. For Cholesky on the inequality-only case this is confirmed by the `chol_reuse` timings.

**Condition number evolution.** As the algorithm converges ($s_i, \lambda_i \to 0$):
- The full KKT matrix remains well-conditioned ($\kappa \sim 10^2$)
- The reduced systems involving $\Lambda^{-1}S$ or $S^{-1}\Lambda$ develop extreme condition numbers ($\kappa > 10^{100}$)
- Strategy 2 (Cholesky on $\hat G$) is immune because $\hat G = G + C\,S^{-1}\Lambda\,C^T$ stays SPD with $\kappa = 1$ when $G = I$

This explains why LDLT on the reduced system needs more iterations under strict tolerance: the accumulated roundoff from solving a $\kappa \sim 10^{100}$ system limits achievable precision to ~$10^{-14}$.

**Normalized time.** Dividing solve time by $\text{dim}^3$ produces roughly constant values across problem sizes, confirming the expected $O(\text{dim}^3)$ complexity of dense solves. The practical savings come entirely from reducing dim, not from a better complexity class.